In [ ]:
import paho.mqtt.client as mqtt
import csv
import re
from datetime import datetime
import time
import os

BROKER_ADDRESS = "127.0.0.1"
MQTT_TOPIC = "imu/data"
SAMPLES_PER_SEC = 5
DURATION_SEC = 4
EXPECTED_MESSAGES = SAMPLES_PER_SEC * DURATION_SEC

FIELDNAMES = ["timestamp","acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]

def collect_session(participant_id, session_id, label, output_dir="data"):
    # prepare folder & filename
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    session_filename = f"{participant_id}_sess{session_id:02d}_{label}_{ts}.csv"
    participant_folder = os.path.join(output_dir, participant_id)
    os.makedirs(participant_folder, exist_ok=True)
    full_path = os.path.join(participant_folder, session_filename)
    
    # write header
    with open(full_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES + ["label"])
        writer.writeheader()
    
    collected = 0
    
    def on_connect(client, userdata, flags, rc):
        if rc == 0:
            client.subscribe(MQTT_TOPIC)
            print("Connected to broker, starting collection")
        else:
            print("Connection failed:", rc)
    
    def on_message(client, userdata, msg):
        nonlocal collected
        payload = msg.payload.decode("utf-8").strip()
        matches = re.findall(r"\[([0-9\.,\- ]+)\]", payload)
        rows = []
        for m in matches:
            vals = [float(x) for x in m.split(",")]
            row = {
                "timestamp": vals[0],
                "acc_x": vals[1],
                "acc_y": vals[2],
                "acc_z": vals[3],
                "gyro_x": vals[4],
                "gyro_y": vals[5],
                "gyro_z": vals[6],
                "label": label
            }
            rows.append(row)
        
        with open(full_path, "a", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            writer.writerows(rows)
        
        collected += 1
        print(f"Collected {collected}/{EXPECTED_MESSAGES} messages (approx)")
        if collected >= EXPECTED_MESSAGES:
            client.disconnect()
    
    client = mqtt.Client()
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER_ADDRESS, 1883, 60)
    client.loop_forever()
    print(f"Session saved: {full_path}")
    return full_path

# Example use:
participant_id = int(input("Enter participant ID: "))
session_id     = int(input("Session number: "))
label          = input("Label w(walking)/f(falling): ")
collect_session(participant_id, session_id, label)



/var/folders/m1/qxpcclg55hg_8gcmt4vpkrq80000gn/T/ipykernel_52994/3584737096.py:66: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


Connected to broker, starting collection
Collected 1/5 messages (approx)
Collected 2/5 messages (approx)
Collected 3/5 messages (approx)
Collected 4/5 messages (approx)
Collected 5/5 messages (approx)
Session saved: data/1/1_sess02_w_20251018_103338.csv


'data/1/1_sess02_w_20251018_103338.csv'